# Add `file_extension` to the `Fabric_Governance` model

The gold table `gold_file_dependencies` has a `file_extension` column, but the
Direct Lake model was created before it existed (Direct Lake doesn't auto-add new
columns). This adds just that one column via TOM — **no model recreate**, so the
existing report and measures stay intact — then reframes so the values load.

Idempotent: re-running is a no-op if the column already exists.

In [ ]:
%pip install -q semantic-link-labs "PyJWT>=2.6.0"

In [ ]:
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

MODEL = "Fabric_Governance"
TABLE = "gold_file_dependencies"

with connect_semantic_model(dataset=MODEL, readonly=False) as tom:
    have = {c.Name for c in tom.all_columns() if c.Parent.Name == TABLE}
    if "file_extension" in have:
        print("file_extension already present -- nothing to do")
    else:
        tom.add_data_column(TABLE, "file_extension",
                            source_column="file_extension", data_type="String")
        print("added file_extension to", TABLE)

# Reframe Direct Lake so the new column's data loads.
labs.refresh_semantic_model(dataset=MODEL)
print("model refreshed")

# Confirm
with connect_semantic_model(dataset=MODEL, readonly=True) as tom:
    cols = sorted(c.Name for c in tom.all_columns() if c.Parent.Name == TABLE)
    print(f"{TABLE} columns:", cols)